##### Basic config

In [ ]:
import pandas as pd
import json

df = pd.read_excel('../data/Elsevier-CS/Texts_3000-lite-abstract.xlsx')
links = df['Pii'].tolist()
abs = df['reserve_4'].tolist()
hts = df['Highlights'].tolist()

link_to_keywords = {}
with open('../data/Elsevier-CS/Keywords.json', 'r') as f:
    link_to_keywords = json.load(f)

keywords = []
for i, link in enumerate(links):
    try:
        keywords.append(link_to_keywords[link])
    except:
        keywords.append([])

print(len(keywords))

2996


In [ ]:
import nltk
import numpy as np

porter = nltk.PorterStemmer()
def stemmer(raw_sequences):
    stemmed_sequences = []

    for i, words in enumerate(raw_sequences):
        new_words = []
        for word in words:
            if type(word) == list:
                word = word[0]
            items = word.split()
            new_word = ' '.join(porter.stem(item) for item in items)
            new_words.append(new_word.strip())
        stemmed_sequences.append(new_words)
    
    return stemmed_sequences


def calPRF(num_c, num_e, num_s):
    F1 = 0.0
    P = float(num_c) / float(num_e) if num_e!=0 else 0.0
    R = float(num_c) / float(num_s) if num_s!=0 else 0.0
    if (P + R == 0.0):
        F1 = 0
    else:
        F1 = 2 * P * R / (P + R)
    return P, R, F1


def getPRF_docwise(references, predictions, log=None):
    P5_list, R5_list, F5_list = [], [], []
    P10_list, R10_list, F10_list = [], [], []
    P15_list, R15_list, F15_list = [], [], []

    assert len(references) == len(predictions), "refs 和 preds 数量必须一致"

    for i in range(len(references)):
        reference = references[i]
        prediction = predictions[i]

        num_s = len(reference)  # 当前文档真实关键词数量

        # --- k = 5 ---
        pred_k = prediction[:5]
        num_e_5 = len(pred_k)
        num_c_5 = sum(1 for cand in pred_k if cand in reference)
        P5, R5, F5 = calPRF(num_c_5, num_e_5, num_s)
        P5_list.append(P5)
        R5_list.append(R5)
        F5_list.append(F5)

        # --- k = 10 ---
        pred_k = prediction[:10]
        num_e_10 = len(pred_k)
        num_c_10 = sum(1 for cand in pred_k if cand in reference)
        P10, R10, F10 = calPRF(num_c_10, num_e_10, num_s)
        P10_list.append(P10)
        R10_list.append(R10)
        F10_list.append(F10)

        # --- k = 15 ---
        pred_k = prediction[:15]
        num_e_15 = len(pred_k)
        num_c_15 = sum(1 for cand in pred_k if cand in reference)
        P15, R15, F15 = calPRF(num_c_15, num_e_15, num_s)
        P15_list.append(P15)
        R15_list.append(R15)
        F15_list.append(F15)

    per_doc_metrics = {
        "F5": F5_list,
        "F10": F10_list,
        "F15": F15_list,
    }
    
    return per_doc_metrics

##### load llm data

In [27]:
method_results = {}

In [ ]:
import re
import pandas as pd

# load llm data
new_df = pd.read_excel('temporary_data/ModelPred/LLM/Claude/Elsevier-LIS-Claude-HA.xlsx')
raw_pred_keywords = new_df['Keywords'].tolist()
pred_keywords = []
regex = r'\d. '
for elem in raw_pred_keywords:
    keyword = []
    if ':' in elem:
        pos = elem.rindex(':')
        elem = elem[pos+1:]
        if ',' in elem:
            keyword = elem.split(',')
        elif '\n' in elem:
            keyword = elem.split('\n')
    if ':' not in elem:
        if ',' in elem:
            keyword = elem.split(',')
        elif '\n' in elem:
            keyword = elem.split('\n')

    new_keyword = []
    for i, word in enumerate(keyword):
        word = word.strip()
        if len(word.split(' ')) >=5:
            continue
        if '- ' in word:
            pos = word.index(' ')
            word = word[pos+1:]
        elif re.search(regex, word):
            pos = re.search(regex, word).span()[1]
            word = word[pos:]

        if word != '':
            if word[-1] == '.':
                new_keyword.append(word[:-1])
            else:
                new_keyword.append(word)

    keyword = [word.strip() for word in new_keyword]
    pred_keywords.append(keyword)

In [ ]:
gold_standards_stem = stemmer(keywords)
pred_candidates_stem = stemmer(pred_keywords)
results = getPRF_docwise(gold_standards_stem, pred_candidates_stem)
method_results['HA'] = results

##### load traditional data

In [ ]:
method_results = {}

In [ ]:
# load traditional method data
import json

with open("temporary_data/ModelPred/PositionRank/keywords/CS_H+FA.json", 'r') as f:
    pred_keywords = json.load(f)

gold_standards_stem = stemmer(keywords)
results = getPRF_docwise(gold_standards_stem, pred_keywords)
method_results['HFA'] = results

##### paired t-test

In [ ]:
from scipy import stats

a_results = method_results['A']

options = method_results.keys()
for option in options:
    if option == 'A':
        continue

    print(option)
    option_results = method_results[option]
    try:
        for score in ['F5', 'F10', 'F15']:
            a_result = a_results[score]
            b_result = option_results[score]
            t_stat, p_value = stats.ttest_rel(a_result, b_result)
            diff = np.mean(b_result) - np.mean(a_result)
            print(score, end=" ")
            print(diff, p_value)
    except:
        continue

H
F5 -0.05664185243624496 2.388401342955594e-72
F10 -0.058147095079671834 1.0779651806010739e-125
F15 -0.048811404366644155 3.1955289194171325e-128
AH
F5 0.0038649175331418595 0.006386674825482902
F10 0.007458907358773859 2.1559879200909853e-13
F15 0.007527078711464921 1.354582987414531e-22
HA
F5 0.0018816334236895216 0.42878862963020437
F10 0.0066106762618778625 1.5170111121283677e-05
F15 0.00790979451298475 9.036884589046769e-14
FA
F5 -0.020213651872530353 1.0553096525281624e-15
F10 -0.019885060422443604 4.0593543540516505e-33
F15 -0.021095492222138884 2.0956002611096795e-63
FAH
F5 -0.012515340319078624 2.8943546583170375e-06
F10 -0.008330857504756026 7.397844228644501e-06
F15 -0.00844152548705987 1.243614080164957e-09
HFA
F5 -0.010972542514598588 5.267678292397682e-05
F10 -0.008364235341872181 8.232533197676942e-06
F15 -0.008504237897815278 1.678688145739115e-09
